# TP 2 - Administrateur HDFS : organiser et contrôler un espace de données

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syhard87/tp_hadloop/blob/claude/tp-hadoop-avances-partie-a-mwgfq7/notebooks/TP2_Administrateur_HDFS.ipynb)

Source : `sources/TP_Hadoop_avances_avec_TP_ecosysteme.pdf`, pages 12 et 13.

**Contexte** : on administre l'espace HDFS d'une entreprise qui reçoit des ventes, des clients, des produits et des logs web.
On crée une organisation `raw/curated/analytics`, on charge les fichiers et on contrôle leur stockage.

**Mode d'emploi** : vérifier que l'environnement est connecté (coche verte en haut à droite), puis exécuter les cellules
**dans l'ordre**. Si la session redémarre, reprendre depuis la cellule 1.

## 1. Récupérer le dépôt (jeux de données + script d'installation)

In [ ]:
!rm -rf /content/tp_hadloop
!git clone -q --depth 1 -b claude/tp-hadoop-avances-partie-a-mwgfq7 https://github.com/Syhard87/tp_hadloop.git /content/tp_hadloop
!ls /content/tp_hadloop/data /content/tp_hadloop/scripts

## 2. Variables d'environnement et installation de Hadoop (HDFS + YARN)
Le script `scripts/setup_hadoop_colab.sh` reprend les cellules 1 à 4 du TP 1, avec la correction pour `root`.
Il ajoute la configuration YARN (`mapred-site.xml`, `yarn-site.xml`) et démarre `start-yarn.sh`.
Il peut être relancé sans risque : il ne reformate jamais un NameNode déjà formaté.
**Durée** : 2 à 4 minutes à la première exécution.

In [ ]:
import os
HADOOP_HOME = "/content/hadoop-3.5.0"
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
if f"{HADOOP_HOME}/bin" not in os.environ["PATH"]:
    os.environ["PATH"] = f"{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:" + os.environ["PATH"]
os.makedirs("/content/preuves_tp2", exist_ok=True)

!bash /content/tp_hadloop/scripts/setup_hadoop_colab.sh

## 3. Jeux de données à préparer
Le fichier `logs_web.csv` est fourni. Les trois autres ont été préparés avec les anomalies demandées par l'énoncé :

| Fichier | Colonnes | Anomalie incluse |
|---|---|---|
| `ventes.csv` | date, pays, produit, quantite, prix | 1 ligne en double (`2026-10-06,BE,P03,2,12.00`) |
| `clients.csv` | client_id, pays, segment | 1 pays manquant (`C006`) |
| `produits.csv` | produit_id, famille, prix_ref | 1 prix invalide (`P07` → `abc`) |
| `logs_web.csv` | timestamp, ip, page, statut | 48 statuts 404 |

La cellule suivante copie les fichiers dans `/content/tp2` (le chemin **local**) et vérifie les anomalies.

In [ ]:
!mkdir -p /content/tp2 && cp /content/tp_hadloop/data/*.csv /content/tp2/
!ls -l /content/tp2

import pandas as pd
ventes   = pd.read_csv("/content/tp2/ventes.csv")
clients  = pd.read_csv("/content/tp2/clients.csv")
produits = pd.read_csv("/content/tp2/produits.csv")
logs     = pd.read_csv("/content/tp2/logs_web.csv")

print("ventes   - lignes en double :", ventes.duplicated().sum())
print(ventes[ventes.duplicated(keep=False)], "\n")
print("clients  - pays manquants   :", clients["pays"].isna().sum())
print(clients[clients["pays"].isna()], "\n")
prix = pd.to_numeric(produits["prix_ref"], errors="coerce")
print("produits - prix invalides   :", prix.isna().sum())
print(produits[prix.isna()], "\n")
print("logs_web - répartition des statuts :")
print(logs["statut"].value_counts().to_string())
print("\nlogs_web - pages en 404 :")
print(logs[logs["statut"] == 404]["page"].value_counts().to_string())

## Mission A - Préparer les services
**Attendu** : `jps` affiche NameNode, DataNode, SecondaryNameNode, ResourceManager et NodeManager ;
`Live datanodes (1)` ; YARN indique `Total Nodes:1` avec un nœud `RUNNING`.

In [ ]:
!jps | tee /content/preuves_tp2/A_jps.txt
!hdfs dfsadmin -report 2>/dev/null | tee /content/preuves_tp2/A_dfsadmin_report.txt
!yarn node -list 2>/dev/null | tee /content/preuves_tp2/A_yarn_nodes.txt

## Mission B - Construire l'arborescence

In [ ]:
%%bash
D=ingestion_date=2026-10-11
hdfs dfs -mkdir -p \
  /data/raw/sales/$D \
  /data/raw/customers/$D \
  /data/raw/products/$D \
  /data/raw/web_logs/$D \
  /data/curated/sales/year=2026/month=10 \
  /data/analytics \
  /data/rejects
hdfs dfs -ls -R /data | tee /content/preuves_tp2/B_arborescence.txt

## Mission C - Charger et inspecter
Questions 10 à 14 : charger les quatre fichiers, lire un extrait, observer la taille logique, les blocs et la réplication,
puis comparer le chemin local et le chemin HDFS.

In [ ]:
%%bash
D=ingestion_date=2026-10-11
# 10. Charger les quatre fichiers
hdfs dfs -put -f /content/tp2/ventes.csv   /data/raw/sales/$D/
hdfs dfs -put -f /content/tp2/clients.csv  /data/raw/customers/$D/
hdfs dfs -put -f /content/tp2/produits.csv /data/raw/products/$D/
hdfs dfs -put -f /content/tp2/logs_web.csv /data/raw/web_logs/$D/

# 11. Vérifier leur présence
hdfs dfs -ls -R /data | tee /content/preuves_tp2/C_ls_R.txt

In [ ]:
%%bash
D=ingestion_date=2026-10-11
# 11. Lire un extrait (le message « Unable to write to output stream » est normal : head ferme le flux)
for f in sales/$D/ventes.csv customers/$D/clients.csv products/$D/produits.csv web_logs/$D/logs_web.csv; do
  echo "--- /data/raw/$f"
  hdfs dfs -cat /data/raw/$f 2>/dev/null | head -n 4
done

# 12. Taille logique (1re colonne) et espace consommé avec la réplication (2e colonne)
echo; echo "=== du -h /data/raw ==="
hdfs dfs -du -h /data/raw | tee /content/preuves_tp2/C_du.txt
hdfs dfs -du -s -h /data/raw | tee -a /content/preuves_tp2/C_du.txt

In [ ]:
%%bash
D=ingestion_date=2026-10-11
# 13. Nombre de blocs et réplication (au moins deux fichiers demandés)
for f in web_logs/$D/logs_web.csv sales/$D/ventes.csv; do
  hdfs fsck /data/raw/$f -files -blocks -locations 2>/dev/null
done | tee /content/preuves_tp2/C_fsck.txt

### 14. Tableau comparatif local / HDFS

In [ ]:
import subprocess, os, math, pandas as pd

D = "ingestion_date=2026-10-11"
fichiers = {
    "ventes.csv":   f"/data/raw/sales/{D}/ventes.csv",
    "clients.csv":  f"/data/raw/customers/{D}/clients.csv",
    "produits.csv": f"/data/raw/products/{D}/produits.csv",
    "logs_web.csv": f"/data/raw/web_logs/{D}/logs_web.csv",
}
lignes = []
for nom, hdfs_path in fichiers.items():
    local = f"/content/tp2/{nom}"
    taille_hdfs, repl, bloc = subprocess.run(
        ["hdfs", "dfs", "-stat", "%b %r %o", hdfs_path],
        capture_output=True, text=True).stdout.split()
    lignes.append({
        "Fichier": nom,
        "Chemin local": local,
        "Taille locale (o)": os.path.getsize(local),
        "Chemin HDFS": hdfs_path,
        "Taille HDFS (o)": int(taille_hdfs),
        "Réplication": int(repl),
        "Taille de bloc": int(bloc),
        "Nb blocs": math.ceil(int(taille_hdfs) / int(bloc)),
        "Lignes (avec en-tête)": sum(1 for _ in open(local)),
    })
tableau = pd.DataFrame(lignes)
tableau.to_csv("/content/preuves_tp2/C_tableau_local_hdfs.csv", index=False)
tableau

## Mission D - Droits et cycle de vie

In [ ]:
%%bash
hdfs dfs -chmod 750 /data/curated
hdfs dfs -chmod 770 /data/analytics
hdfs dfs -touchz /data/analytics/_READY

hdfs dfs -ls -d /data/raw /data/curated /data/analytics /data/rejects
echo
hdfs dfs -stat "%n %b %r" /data/analytics/_READY
hdfs dfs -stat "%n %b %r" /data/raw/web_logs/ingestion_date=2026-10-11/logs_web.csv
hdfs dfs -test -e /data/analytics/_READY && echo "_READY présent : les données analytics peuvent être consommées"

## Sauvegarder les preuves dans l'espace partagé (Drive)
Crée `MyDrive/TP_Hadoop_partage/preuves_tp2`. Partager ce dossier depuis Drive (clic droit → *Partager*).

In [ ]:
!hdfs dfs -ls -R /data > /content/preuves_tp2/D_arborescence_finale.txt
from google.colab import drive
import shutil
drive.mount('/content/drive')
dest = "/content/drive/MyDrive/TP_Hadoop_partage/preuves_tp2"
shutil.copytree("/content/preuves_tp2", dest, dirs_exist_ok=True)
print("Preuves copiées dans", dest, ":", sorted(os.listdir(dest)))

---
# Réponses

## Mission A
- **Présence des services** : `jps` affiche **NameNode**, **DataNode** et **SecondaryNameNode** pour HDFS,
  ainsi que **ResourceManager** et **NodeManager** pour YARN.
- **DataNodes vivants** : `Live datanodes (1)`, `Dead datanodes (0)`. En mode pseudo-distribué, un seul DataNode tourne sur la machine Colab.
- **HDFS ou YARN ?** **HDFS** est la couche de **stockage**. Le NameNode garde les métadonnées : arborescence, droits,
  liste des blocs et emplacement de chaque bloc. Les DataNodes stockent physiquement les blocs. **YARN** est la couche de
  **gestion des ressources et d'exécution des traitements**. Le ResourceManager alloue le CPU et la mémoire aux applications,
  et les NodeManagers lancent les conteneurs sur chaque nœud. HDFS répond à « où sont les données ? », YARN à « qui calcule,
  avec quelles ressources ? ». On peut lire et écrire dans HDFS sans YARN, mais un job MapReduce ou Spark sur le cluster a besoin de YARN.

## Mission B
| Zone | Rôle | Justification |
|---|---|---|
| `raw/` | Données **brutes**, telles que reçues | Copie fidèle et immuable de la source : on peut toujours rejouer un traitement ou auditer une donnée. On n'y corrige rien. |
| `curated/` | Données **nettoyées et validées** | Doublons supprimés, types vérifiés, partitionnement métier (`year=/month=`). C'est la source fiable des traitements. |
| `analytics/` | **Résultats** prêts à consommer | Agrégats et indicateurs (CA par pays, top pages…) lus par les analystes et les tableaux de bord. |
| `rejects/` | Lignes **rejetées** | Lignes invalides isolées (doublon, pays manquant, prix `abc`) pour qu'elles ne polluent pas `curated` et restent analysables. |

**Rôle de `ingestion_date`** dans `raw` : c'est une **partition technique** qui indique **le jour de réception** des
données, pas leur date métier. Elle permet :
- de rejouer ou supprimer un chargement précis sans toucher aux autres ;
- d'éviter qu'une nouvelle livraison écrase la précédente ;
- de tracer l'origine d'une donnée ;
- d'appliquer une règle de rétention par date (archiver les `ingestion_date` de plus de N jours).

La forme `cle=valeur` est reconnue par Hive et Spark comme une colonne de partition.

## Mission C
- **10-11** : les quatre fichiers sont présents dans `hdfs dfs -ls -R /data`, et `hdfs dfs -cat … | head` affiche
  l'en-tête suivi des premières lignes.
- **12. Taille logique** : `ventes.csv` 395 o, `clients.csv` 229 o, `produits.csv` 184 o, `logs_web.csv` 9 766 o,
  soit environ 10,3 K pour `/data/raw`. Dans `du`, la 1re colonne donne la taille logique et la 2e l'espace consommé en
  tenant compte de la réplication. Ici les deux sont égales, car la réplication vaut 1.
- **13. Blocs et réplication** : chaque fichier tient dans **1 seul bloc**. Par exemple, `fsck` affiche `len=9766 Live_repl=1`
  pour `logs_web.csv`, et la réplication est de **1**. Un bloc HDFS n'occupe sur disque que la taille réelle des données :
  un fichier de 9 766 o ne consomme pas 128 Mo. En revanche, chaque fichier et chaque bloc coûtent une entrée en
  mémoire dans le NameNode, d'où le problème des petits fichiers (TP 4).
- **14. Local ou HDFS** :

| | Chemin local | Chemin HDFS |
|---|---|---|
| Exemple | `/content/tp2/logs_web.csv` | `/data/raw/web_logs/ingestion_date=2026-10-11/logs_web.csv` |
| Système de fichiers | Disque de la VM Colab (Linux) | Système de fichiers distribué (`hdfs://localhost:9000`) |
| Commandes | `ls`, `cat`, `du` | `hdfs dfs -ls`, `-cat`, `-du` |
| Stockage | Un fichier sur un disque | Découpé en blocs, répliqué, métadonnées dans le NameNode |
| Taille | 9 766 o | 9 766 o (identique) |

Le contenu est identique, mais le fichier HDFS n'est pas visible avec un `ls` Linux. Ses blocs sont stockés sous
`/content/hdfs/data` sous forme de fichiers `blk_…`.

## Mission D
- **750 sur `/data/curated`** (`drwxr-x---`) : le **propriétaire** a lecture, écriture et parcours (7). Le **groupe** peut lire
  et parcourir, mais pas écrire (5). Les **autres** n'ont aucun accès (0). Seul le pipeline propriétaire écrit les données
  validées ; l'équipe peut les lire sans risquer de les modifier.
- **770 sur `/data/analytics`** (`drwxrwx---`) : le propriétaire **et le groupe** ont tous les droits (7). Les autres n'ont
  aucun accès. C'est un espace de travail collaboratif : les analystes du groupe créent et suppriment leurs résultats.
- **Fichier marqueur `_READY`** : c'est un fichier vide (0 octet) écrit **en dernier**, une fois toutes les données produites.
  Les consommateurs (job aval, ordonnanceur, tableau de bord) attendent sa présence (`hdfs dfs -test -e …/_READY`) avant
  de lire. Cela évite de lire un dossier incomplet pendant l'écriture. Le principe est le même que `_SUCCESS`, écrit par MapReduce et Spark.
- **Règle d'archivage ou de suppression de `raw`** :
  1. Conserver les partitions `ingestion_date` des **30 derniers jours** dans `raw`.
  2. Au-delà, une fois la partition traitée vers `curated` et `_READY` présent, l'**archiver** :
     compression, archive HAR ou stockage froid avec `hdfs storagepolicies -setStoragePolicy -policy COLD`.
  3. **Supprimer** les archives après la durée légale de conservation (par exemple 1 an).

  Ne jamais supprimer une partition `raw` qui n'a pas encore été traitée.